# Part 1 — Embedding Model Identification

The attacker has a stolen embedding and wants to know which model produced it.

The target database contains a known entry `TEST` (text: `"test"`).  
For each candidate model, encode `"test"` and compare cosine similarity against the stolen `TEST` embedding.  
The model with cosine closest to 1.0 is the likely source.

In [ ]:
import gc
import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoConfig
from tqdm.notebook import tqdm

torch.set_num_threads(1)  # M1 Accelerate/OpenMP segfault fix for CPU inference

TEST_EMBEDDING_ID = 'TEST'
TEST_CONTENT      = 'test'
TARGET_DATA_FILE  = 'data/sentences_target_text_db.parquet'
TARGET_INDEX_FILE = 'data/sentences_target_vector_db.index'
DEVICE = 'cpu'

CANDIDATE_MODELS = [
    'all-MiniLM-L6-v2',
    'paraphrase-multilingual-MiniLM-L12-v2',
    'BAAI/bge-small-en-v1.5',
    'all-mpnet-base-v2',
    'BAAI/bge-base-en-v1.5',
    'intfloat/multilingual-e5-small',
    'paraphrase-multilingual-mpnet-base-v2',
    'all-MiniLM-L12-v2',
    'all-distilroberta-v1',
    'multi-qa-mpnet-base-dot-v1',
    'multi-qa-MiniLM-L6-cos-v1',
    'msmarco-distilbert-base-v4',
    'intfloat/e5-base-v2',
    'sentence-transformers/gtr-t5-base',
]

In [ ]:
target_df    = pd.read_parquet(TARGET_DATA_FILE)
target_index = faiss.read_index(TARGET_INDEX_FILE)
all_emb      = np.zeros((target_index.ntotal, target_index.d), dtype=np.float32)
target_index.reconstruct_n(0, target_index.ntotal, all_emb)

test_row = target_df[target_df['target_id'] == TEST_EMBEDDING_ID].iloc[0]
test_emb = all_emb[test_row['id']]

print(f'Leaked embedding loaded - dim={test_emb.shape[0]}')

In [ ]:
def resolve_repo_id(name: str) -> str:
    return name if '/' in name else f'sentence-transformers/{name}'

compatible_models = []
for name in CANDIDATE_MODELS:
    config = AutoConfig.from_pretrained(resolve_repo_id(name), trust_remote_code=False)
    dim = next(
        (getattr(config, attr) for attr in ('sentence_embedding_dimension', 'embedding_size', 'hidden_size', 'd_model')
         if isinstance(getattr(config, attr, None), int)),
        None,
    )
    match = dim == test_emb.shape[0]
    if match:
        compatible_models.append(name)
    print(f'{name}: dim={dim} | {"ok" if match else "skip"}')

print(f'\n{len(compatible_models)}/{len(CANDIDATE_MODELS)} compatible')

In [ ]:
results = []
for model_name in tqdm(compatible_models):
    model    = SentenceTransformer(model_name, device=DEVICE)
    cand_emb = model.encode([TEST_CONTENT], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)[0].astype(np.float32)
    del model
    gc.collect()

    cosine = float(np.dot(test_emb, cand_emb))
    results.append((model_name, cosine))

In [ ]:
results.sort(key=lambda x: x[1] if not np.isnan(x[1]) else float('-inf'), reverse=True)
for rank, (name, score) in enumerate(results, 1):
    print(f'{rank} - {score:.4f} | {name}')

In [ ]:
del test_emb, all_emb, target_index, target_df, compatible_models, results
gc.collect()
print('notebook resources released')